# Proyecto 1 – Saber 11  
## Tarea 2: Selección, limpieza y alistamiento (Ingeniería de Datos)

Este notebook realiza la **limpieza completa** y además filtra **Bogotá** de forma robusta (sin importar mayúsculas/minúsculas, tildes o comillas en el texto).

**Entrada:** `proyectoSaber11.csv` (archivo grande con 51 columnas)  
**Salida principal:** `saber11_bogota_limpio.csv` (listo para análisis)  
**Salida para Excel:** `saber11_bogota_limpio_excel.csv` (separador `;`)


## 0) Configuración

- Pon este notebook en la misma carpeta donde está `proyectoSaber11.csv`.
- Si el archivo tiene otro nombre o está en otra ruta, cambia `INPUT_FILE`.


In [ ]:
import pandas as pd
import numpy as np
import re

INPUT_FILE = "proyectoSaber11.csv"   # <-- cambia si tu archivo tiene otro nombre/ruta
OUTPUT_MAIN = "saber11_bogota_limpio.csv"
OUTPUT_EXCEL = "saber11_bogota_limpio_excel.csv"


## 1) Carga del dataset

Usamos `low_memory=False` para evitar inferencias inconsistentes en archivos grandes.

In [ ]:
df = pd.read_csv(INPUT_FILE, low_memory=False)
print("Shape original (filas, columnas):", df.shape)
df.head(3)

## 2) Normalización de texto para detectar Bogotá (robusto)

Vamos a construir una versión normalizada de los campos de ubicación:

1. Convertimos a texto
2. Quitamos comillas dobles
3. Quitamos tildes (ÁÉÍÓÚ → AEIOU)
4. Pasamos a minúsculas
5. Quitamos espacios extra

Luego filtramos todas las filas donde el **departamento** o el **municipio** contengan la palabra `bogota`.

In [ ]:
def normalize_text(s: pd.Series) -> pd.Series:
    s = s.astype("string")
    s = s.str.replace('"', '', regex=False)  # quita comillas dobles
    # quita tildes solo para vocales (suficiente para Bogotá)
    s = s.str.translate(str.maketrans("ÁÉÍÓÚáéíóú", "AEIOUaeiou"))
    s = s.str.strip().str.lower()
    # normaliza espacios múltiples
    s = s.str.replace(r"\s+", " ", regex=True)
    return s

# Columnas de ubicación (colegio)
for col in ["cole_depto_ubicacion", "cole_mcpio_ubicacion"]:
    if col in df.columns:
        df[col + "_norm"] = normalize_text(df[col])
    else:
        print(f"No existe la columna {col} en este archivo.")

# Filtro Bogotá (por depto o por municipio)
mask_bogota = False
if "cole_depto_ubicacion_norm" in df.columns:
    mask_bogota = mask_bogota | df["cole_depto_ubicacion_norm"].str.contains("bogota", na=False)
if "cole_mcpio_ubicacion_norm" in df.columns:
    mask_bogota = mask_bogota | df["cole_mcpio_ubicacion_norm"].str.contains("bogota", na=False)

df_bog = df.loc[mask_bogota].copy()
print("Shape después de filtrar Bogotá:", df_bog.shape)

# Verifica valores originales más comunes (rápido)
if "cole_depto_ubicacion" in df_bog.columns:
    display(df_bog["cole_depto_ubicacion"].value_counts().head(10))
if "cole_mcpio_ubicacion" in df_bog.columns:
    display(df_bog["cole_mcpio_ubicacion"].value_counts().head(10))


## 3) Selección de columnas relevantes (alineado con las preguntas de negocio)

Para tus preguntas (estrato/hogar, ubicación, tipo de colegio), estas columnas son las más útiles.


In [ ]:
keep_cols = [
    # contexto
    "periodo",
    # ubicación del colegio
    "cole_area_ubicacion", "cole_depto_ubicacion", "cole_mcpio_ubicacion",
    # características del colegio
    "cole_naturaleza", "cole_calendario", "cole_jornada", "cole_caracter",
    "cole_bilingue", "cole_genero",
    # hogar / estrato
    "fami_estratovivienda", "fami_personashogar", "fami_cuartoshogar",
    "fami_tieneinternet", "fami_tienecomputador",
    # puntajes
    "punt_global", "punt_matematicas", "punt_lectura_critica",
    "punt_c_naturales", "punt_sociales_ciudadanas", "punt_ingles",
    "desemp_ingles",
    # (opcional) identificador del estudiante si quieren deduplicar a nivel estudiante
    "estu_consecutivo"
]

existing = [c for c in keep_cols if c in df_bog.columns]
missing = [c for c in keep_cols if c not in df_bog.columns]

print("Columnas seleccionadas:", len(existing))
if missing:
    print("Columnas no encontradas (se omiten):", missing)

df_sel = df_bog[existing].copy()
df_sel.head(3)

## 4) Limpieza de texto general (quitar comillas y espacios)

Aplicamos esto a todas las columnas tipo texto del subconjunto seleccionado.

In [ ]:
obj_cols = df_sel.select_dtypes(include="object").columns
for c in obj_cols:
    df_sel[c] = (
        df_sel[c]
        .astype("string")
        .str.strip()
        .str.replace('"', "", regex=False)
        .str.strip()
    )

print("Ejemplo depto (limpio):", df_sel["cole_depto_ubicacion"].dropna().unique()[:10] if "cole_depto_ubicacion" in df_sel.columns else "N/A")


## 5) Normalización SI/NO (internet y computador)

- Solo dejamos `SI` o `NO`
- Cualquier otro valor → `NA`

In [ ]:
def normalizar_si_no(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip().upper()
    if s in ["SI", "SÍ"]:
        return "SI"
    if s == "NO":
        return "NO"
    return pd.NA

for col in ["fami_tieneinternet", "fami_tienecomputador"]:
    if col in df_sel.columns:
        df_sel[col] = df_sel[col].apply(normalizar_si_no)

for col in ["fami_tieneinternet", "fami_tienecomputador"]:
    if col in df_sel.columns:
        print("\nFrecuencias:", col)
        display(df_sel[col].value_counts(dropna=False).head(10))


## 6) Feature engineering: estrato numérico

Creamos:
- `estrato_num` (1–6)
- `estrato_sin` (True si dice SIN ESTRATO)

In [ ]:
if "fami_estratovivienda" in df_sel.columns:
    estr = df_sel["fami_estratovivienda"].astype("string").str.upper().str.strip()
    df_sel["estrato_num"] = estr.str.extract(r"ESTRATO\s+([1-6])", expand=False).astype("Float64")
    df_sel["estrato_sin"] = estr.eq("SIN ESTRATO")
    display(df_sel[["fami_estratovivienda","estrato_num","estrato_sin"]].head(10))
    display(df_sel["estrato_num"].value_counts(dropna=False).sort_index())


## 7) Conversión a numérico de puntajes y conteos

Convertimos a numérico:
- puntajes
- número de personas y cuartos
- periodo

In [ ]:
num_cols = [
    "periodo",
    "punt_global","punt_matematicas","punt_lectura_critica",
    "punt_c_naturales","punt_sociales_ciudadanas","punt_ingles",
    "fami_personashogar","fami_cuartoshogar"
]
for c in num_cols:
    if c in df_sel.columns:
        df_sel[c] = pd.to_numeric(df_sel[c], errors="coerce")

df_sel[[c for c in ["punt_global","punt_matematicas","punt_lectura_critica","punt_ingles"] if c in df_sel.columns]].describe()


## 8) Reglas mínimas de calidad

- Eliminamos filas sin `punt_global`
- Filtramos puntajes fuera de rango (0–500)
- Eliminamos duplicados exactos

In [ ]:
before = df_sel.shape[0]

if "punt_global" in df_sel.columns:
    df_sel = df_sel.dropna(subset=["punt_global"])
    df_sel = df_sel[(df_sel["punt_global"] >= 0) & (df_sel["punt_global"] <= 500)]

# Duplicados exactos
df_sel = df_sel.drop_duplicates()

after = df_sel.shape[0]
print("Filas antes:", before)
print("Filas después:", after)
print("Removidas:", before - after)


## 9) Guardado de salidas

- `OUTPUT_MAIN`: CSV estándar (coma) para trabajar en Python
- `OUTPUT_EXCEL`: CSV con `;` para que Excel lo abra en columnas correctamente

In [ ]:
df_sel.to_csv(OUTPUT_MAIN, index=False, encoding="utf-8")
df_sel.to_csv(OUTPUT_EXCEL, index=False, sep=";", encoding="utf-8")

print("Guardado:", OUTPUT_MAIN)
print("Guardado (Excel):", OUTPUT_EXCEL)
print("Shape final:", df_sel.shape)
